In [7]:
import pandas as pd
import numpy as np
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report
from pathlib import Path
from sklearn.metrics import f1_score,accuracy_score



In [3]:
DATA_DIR = Path("../data/")
OUT_DIR = (DATA_DIR / "processed")

def load_split(name):
    data = np.load(OUT_DIR / f"{name}.npz", allow_pickle=True)
    return normalize(data["X"]), data["y"]
def normalize(X):
    X = X.astype(np.float32)
    mins = X.min(axis=(1,2), keepdims=True)
    maxs = X.max(axis=(1,2), keepdims=True)
    return (X - mins) / (maxs - mins + 1e-8)

x_train, y_train = load_split("train")
x_val, y_val = load_split("val")
x_test, y_test = load_split("test")

In [9]:

DC = DummyClassifier(strategy='stratified', random_state=0)

DC.fit(x_train, y_train)

val_predict = DC.predict(x_val)

test_predict = DC.predict(x_test)
print(classification_report(y_val, val_predict))
print(classification_report(y_test, test_predict))


              precision    recall  f1-score   support

      glioma       0.22      0.22      0.22       181
  meningioma       0.20      0.21      0.20       179
     notumor       0.21      0.19      0.20       165
   pituitary       0.21      0.22      0.22       176

    accuracy                           0.21       701
   macro avg       0.21      0.21      0.21       701
weighted avg       0.21      0.21      0.21       701

              precision    recall  f1-score   support

      glioma       0.24      0.26      0.25       386
  meningioma       0.26      0.28      0.27       398
     notumor       0.23      0.20      0.21       400
   pituitary       0.28      0.28      0.28       400

    accuracy                           0.25      1584
   macro avg       0.25      0.25      0.25      1584
weighted avg       0.25      0.25      0.25      1584



In [8]:
results = []
for seed in range(5):
    DC = DummyClassifier(strategy='stratified', random_state=seed)
    DC.fit(x_train, y_train)

    val_pred = DC.predict(x_val)
    test_pred = DC.predict(x_test)

    results.append({
        "seed": seed,
        "val_acc": accuracy_score(y_val, val_pred),
        "val_f1": f1_score(y_val, val_pred, average="macro"),
        "test_acc": accuracy_score(y_test, test_pred),
        "test_f1": f1_score(y_test, test_pred, average="macro"),
    })
seeds_df = pd.DataFrame(results)
print(seeds_df.round(3))
print(seeds_df.drop(columns="seed").agg(["mean", "std"]).round(3))

   seed  val_acc  val_f1  test_acc  test_f1
0     0    0.211   0.211     0.252    0.251
1     1    0.270   0.270     0.232    0.232
2     2    0.241   0.241     0.251    0.250
3     3    0.271   0.271     0.272    0.272
4     4    0.244   0.244     0.247    0.247
      val_acc  val_f1  test_acc  test_f1
mean    0.247   0.247     0.251    0.250
std     0.025   0.025     0.014    0.014


### Final Check

In [10]:
train_check = pd.read_csv(DATA_DIR / "train.csv")
val_check = pd.read_csv(DATA_DIR / "val.csv")
test_check = pd.read_csv(DATA_DIR / "test.csv")
splits = {"train": train_check, "val": val_check, "test": test_check}


for name,df in splits.items():
    print(f"--- {name} ({len(df)} images) ---")
    counts = df["Labels"].value_counts()
    props = df["Labels"].value_counts(normalize=True).round(3)
    print(pd.DataFrame({"count": counts, "proportion": props}))
    print()

# Path overlap between splits (all should be 0)
train_paths = set(train_check["Path"])
val_paths = set(val_check["Path"])
test_paths = set(test_check["Path"])
print("train & val overlap: ", len(train_paths & val_paths))
print("train & test overlap:", len(train_paths & test_paths))
print("val & test overlap:  ", len(val_paths & test_paths))

--- train (4728 images) ---
            count  proportion
Labels                       
glioma       1219       0.258
meningioma   1207       0.255
pituitary    1186       0.251
notumor      1116       0.236

--- val (701 images) ---
            count  proportion
Labels                       
glioma        181       0.258
meningioma    179       0.255
pituitary     176       0.251
notumor       165       0.235

--- test (1584 images) ---
            count  proportion
Labels                       
notumor       400       0.253
pituitary     400       0.253
meningioma    398       0.251
glioma        386       0.244

train & val overlap:  0
train & test overlap: 0
val & test overlap:   0
